<a href="https://colab.research.google.com/github/iliaxant/Pattern_Recognition_Final_Project/blob/main/Final_Project_PR_58545.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Τελική Εργασία Εξαμήνου**

## Αναγνώριση Προτύπων - Ακαδημαϊκό έτος 2025-2026

## Ηλίας Ξανθόπουλος 58545

## GitHub Repo: https://github.com/iliaxant/Pattern_Recognition_Final_Project

---

1) Χειροκίνητο ανέβασμα του αρχείου *Final_Project_data.zip*.

2) Unziping του αρχείου *Final_Project_data.zip*:

In [2]:
import zipfile

zip_path = '/content/Final_Project_data.zip'
zip_ref = zipfile.ZipFile(zip_path, 'r')
zip_ref.extractall('/content')
zip_ref.close()

print("Data unzipped successfully to /content directory.")

Data unzipped successfully to /content directory.


3) Εγκατάσταση και φόρτωση των απαραίτητων βιβλιοθηκών.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import random

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

4) Ορισμός συνάρτησης αρχικοποίησης όλων των χρησιμοποιούμενων random state machines για reproducibility.

In [4]:
def set_seed(seed=15):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

---

## **Μέρος 1ο – Ανίχνευση Αστοχίας σε Βιομηχανική Διαδικασία**

### **Επισκόπηση Δεδομένων**


Ορισμός συνάρτησης ανάλυσης δεδομένων ενός dataframe.

Η συνάρτηση **dataframe_analysis()** δέχεται ως παραμέτρους εισόδου:

*   **df:** το pandas dataframe προς ανάλυση.
*   **label_flag=False:** σημαία που δηλώνει αν υπάρχει στήλη ετικετών στο dataframe.
*   **label_col_name=' '** string που αντιστοιχεί στο όνομα της στήλης ετικετών,εφόσον υπάρχει.

Οι πληροφορίες η συνάρτηση που δίνει για τα δεδομένα αφορούν:

*   Αναλογία των κλάσεων (εφόσον label_flag=Τrue)
*   Missing Values
*   Διπλότυπα δείγματα
*   Features με σταθερή τιμή
*   Features με σχεδόν σταθερή τιμή (πλήθος σταθερών τιμών ίσο ή μεγαλύτερο του 99% των δειγμάτων)

In [47]:
def dataframe_analysis(df, label_flag=False, label_col_name=''):

    print(df.info())

    if label_flag:
        print('\n' + 15 * "-" + " Class Info " + 15 * "-" + '\n')
        print(df[label_col_name].value_counts())
        print()
        print(df[label_col_name].value_counts(normalize=True).map('{:.1%}'.format))
        print('\n' + 42 * "-")


    print('\n' + 15 * "-" + " Missing Values " + 15 * "-" + '\n')

    missing_per_feat = df.isnull().sum()
    missing_per_feat_percent = (df.isnull().mean() * 100).round(2)

    missing_table = pd.DataFrame({
        'Missing Count': missing_per_feat,
        'Percent (%)': missing_per_feat_percent
    })

    missing_table = missing_table.sort_values(by='Percent (%)', ascending=False)

    print(f'Missing Values Table (Descending):')
    print(missing_table)
    print()

    print(f'Τop 20 Missing Values Table:')
    print(missing_table.head(20))
    print()

    if label_flag:

        classes = df[label_col_name].unique()
        for cls in sorted(classes):

            class_subset = df[df[label_col_name] == cls]
            total_missing_class = class_subset.isnull().sum().sum()
            total_cells_class = class_subset.size
            percent_missing_class = (total_missing_class / total_cells_class) * 100

            print(f"Missing values of Class {cls}: {total_missing_class} / {total_cells_class} "
                  f"-> {percent_missing_class:.2f}%")


    total_cells = df.size
    total_missing = missing_per_feat.sum()
    percent_missing = (total_missing / total_cells) * 100

    print(f"\nMissing values of set: {total_missing} / {total_cells} "
          f"-> {percent_missing:.2f}%")

    print('\n' + 46 * "-")


    print('\n' + 15 * "-" + " Duplicates " + 15 * "-" + '\n')
    dupes = df.duplicated()
    print(f'Dublicates List:')
    print(dupes)

    print(f"\nTotal dublicate samples: {dupes.sum()}")
    print('\n' + 42 * "-")


    print('\n' + 15 * "-" + " Constant Features " + 15 * "-" + '\n')
    constant_feat = [feat for feat in df.columns if df[feat].nunique() <= 1]
    print(f"Constant Features List: {constant_feat}")
    print(f'\nTotal constant features: {len(constant_feat)}')
    print('\n' + 48 * "-")


    print('\n' + 15 * "-" + " Quasi-Constant Features " + 15 * "-" + '\n')

    quasi_constant_feats = []
    for feat in df.columns:

        if (feat == label_col_name) or (feat in constant_feat): continue

        top_val_freq = df[feat].value_counts(normalize=True, dropna=False).values[0]

        if top_val_freq >= 0.99:
            quasi_constant_feats.append(feat)

    print(f"Quasi-Constant (>=99% but not 100% of values) Feature List: {quasi_constant_feats}")
    print(f'\nTotal quasi-constant features: {len(quasi_constant_feats)}')

    print('\n' + 55 * "-")



Φόρτωση των training και testing δεδομένων (*Training_data_manifacturing.csv* και *Test_data_manifacturing.csv* αρχεία αντίστοιχα). Ονομασία της κάθε στήλης (καθαρά προαιρετικό) και εκτύπωση χρήσιμων πληροφοριών για τα training και testing δεδομένα μέσω της **dataframe_analysis()**.

In [48]:
columns = [f"Feat {i}" for i in range(474)] + ["y"]

class_column = columns[-1]
data_columns = columns[0:474]

df_train = pd.read_csv("Training_data_manifacturing.csv", header=None, names=columns)

print('\n' + 25 * "=" + " Training Data " + 25 * "=" + '\n')
dataframe_analysis(df_train, 1, class_column)
print('\n' + 65 * "=" + '\n\n')



========================= Training Data =========================

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1254 entries, 0 to 1253
Columns: 475 entries, Feat 0 to y
dtypes: float64(474), int64(1)
memory usage: 4.5 MB
None

--------------- Class Info ---------------

y
0    1170
1      84
Name: count, dtype: int64

y
0    93.3%
1     6.7%
Name: proportion, dtype: object

------------------------------------------

--------------- Missing Values ---------------

Missing Values Table (Descending):
          Missing Count  Percent (%)
Feat 148           1153        91.95
Feat 247           1153        91.95
Feat 149           1153        91.95
Feat 248           1153        91.95
Feat 401           1077        85.89
...                 ...          ...
Feat 418              0         0.00
Feat 459              0         0.00
Feat 417              0         0.00
Feat 18               0         0.00
y                     0         0.00

[475 rows x 2 columns]

Τop 20 Missing Value

Φόρτωση των testing δεδομένων (*Test_data_manifacturing.csv* αρχείο) και εκτύπωση χρήσιμων πληροφοριών για τα δεδομένα μέσω της **dataframe_analysis()**.

**ΣΧΟΛΙΟ:** Αν και εφαρμόζεται η συνάρτηση επισκόπησης και στα testing δεδομένα, παρακάτω δεν πρέπει γίνει καμία επεξεργασία του testing set με βάση τις πληροφορίες που δίνει η συνάρτηση. Ό,τι επεξεργασία ακολουθήσει πάνω στα testing δεδομένα πρέπει να γίνει με γνώμονα **ΜΟΝΟ** το training set. Η χρήση της συνάρτησης επισκόπησης στα testing δεδομένα γίνεται καθαρά για την κατανόηση του προβλήματος προς επίλυση.

In [49]:
df_test = pd.read_csv("Test_data_manifacturing.csv", header=None, names=data_columns)

print('\n' + 25 * "=" + " Testing Data " + 25 * "=" + '\n')
dataframe_analysis(df_test)
print('\n' + 64 * "=" + '\n\n')



========================= Testing Data =========================

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 313 entries, 0 to 312
Columns: 474 entries, Feat 0 to Feat 473
dtypes: float64(473), int64(1)
memory usage: 1.1 MB
None

--------------- Missing Values ---------------

Missing Values Table (Descending):
          Missing Count  Percent (%)
Feat 248            276        88.18
Feat 247            276        88.18
Feat 149            276        88.18
Feat 148            276        88.18
Feat 401            264        84.35
...                 ...          ...
Feat 469              0         0.00
Feat 470              0         0.00
Feat 471              0         0.00
Feat 472              0         0.00
Feat 473              0         0.00

[474 rows x 2 columns]

Τop 20 Missing Values Table:
          Missing Count  Percent (%)
Feat 248            276        88.18
Feat 247            276        88.18
Feat 149            276        88.18
Feat 148            276        88

Παρατηρούνται δύο πολύ σημαντικά προβλήματα στα δεδομένα.

Το πρώτο είναι τα missing values. Τόσο στα training όσο και στα testing δεδομένα υπάρχει ένα ποσοστό ~5.54% των κελιών το οποίο δεν έχει καθόλου τιμή. Αν και φαίνεται αρκετά μικρό το ποσοστό, οι missing τιμές είναι συγκεντρωμένες σε υψηλά ποσοστά σε συγκεκριμένα χαρακτηριστικά, με κάποια από αυτά να φτάνουν missing ποσοστά εώς και 91.95% στο training και 88.18% στο testing. Παρόλα αυτά, όπως φαίνεται στις πληροφορίες για τα training δεδομένα, δεν υπάρχουν καθόλου NaN κελία στην στήλη των ετικετών, το οποίο σημαίνει ότι δεν απαραίτητο να απορριφθούν δείγματα, αλλά πρέπει οπωσδήποτε να επιλεχθεί μια στρατηγική αντιμετώπισης των missing values.

Το δεύτερο πρόβλημα αφορά την κατανομή των κλάσεων και είναι η ανισότητα. Όπως φαίνεται παραπάνω, η συντριπτική πλειοψηφία των δειγμάτων του training dataframe ανήκουν στην κλάση 0 και μόνο το 6.7% των training δεδομένων αντιστοιχούν στην κλάση 1. Παρόλο που δεν υπάρχουν ετικέτες για επιβεβαίωση, είναι εκ των προτέρων γνωστό ότι παρόμοια αναλογία υπάρχει και στα testing δεδομένα. Επομένως, πρέπει να εφαρμοστούν οι κατάλληλες τεχνικές για την διαχείρηση της ανισότητας κλάσεων.

Και τα δύο προβλήματα επιλύονται σε παρακάτω cells.

Επίσης σημαντική παρατήρηση είναι η ύπαρξη σχεδόν σταθερών χαρακτηριστικών στo training set (και σταθερών στο testing set). Αυτή η σταθερότητα καθιστά τα χαρακτηριστικά redundant και οδηγούν σε πιο περίπλοκους τους υπολογισμούς χωρίς κάποια πρακτική αξία. Επομένως, αυτά τα χαρακτηριστικά είναι αναγκαίο να αφαιρεθούν.

**ΣΧΟΛΙΟ:** Στα παραπάνω μηνύματα για τα δεδομένα υπάρχει ένα discrepancy ως προς τον τύπο των δεδομένων.

Παρατηρούμε ότι στα training δεδομένα όλες οι στήλες είναι float64 εκτός από την στήλη των ετικετών που είναι int64. Στα testing δεδομένα που υπάρχουν τα ίδια χαρακτηριστικά αλλά δεν υπάρχει η στήλη ετικετών, πάλι βλέπουμε όλες τις στήλες να είναι float64, εκτός από μία που είναι int64.

Αυτό εν τέλει δεν είναι τυχαίο, διότι αυτόματα η ύπαρξη missing τιμών σε μία στήλη την μετατρέπει σε τύπο δεδομένων float64. Επομένως, στην περίπτωση των training δεδομένων, όλες οι στήλες χαρακτηριστικών τύπου int64 έχουν NaN τιμές, οπότε μετατρέπεται η στήλη σε float64, ενώ στα testing δεδομένα υπάρχει μια στήλη int64 που προηγουμένως είχε missing τιμές, αλλά τώρα δεν έχει, οπότε παραμένει int64.

Αυτό επιβεβαιώνεται τόσο από το παρακάτω code cell, όσο και από μια γρήγορη επισκόπηση των τιμών των δύο *.csv* αρχείων.




In [50]:
print("*** Ignore 'dtype' lines ***\n")

int_cols_test = df_test.select_dtypes(include=['int64']).columns
print(f"Columns of int64 data type: {(list(int_cols_test))}")

print(f"\n--- Info for '{int_cols_test}' on Train data frame ---")
print(f"Data type: {df_train[int_cols_test].dtypes}")
print(f"Num of missing values: {df_train[int_cols_test].isnull().sum()}")

print(f"\n--- Info for '{int_cols_test}' on Test data frame ---")
print(f"Data type: {df_test[int_cols_test].dtypes}")
print(f"Num of missing values: {df_test[int_cols_test].isnull().sum()}")

*** Ignore 'dtype' lines ***

Columns of int64 data type: ['Feat 128']

--- Info for 'Index(['Feat 128'], dtype='object')' on Train data frame ---
Data type: Feat 128    float64
dtype: object
Num of missing values: Feat 128    5
dtype: int64

--- Info for 'Index(['Feat 128'], dtype='object')' on Test data frame ---
Data type: Feat 128    int64
dtype: object
Num of missing values: Feat 128    0
dtype: int64


### **Προεπεξεργασία Δεδομένων**

Μετά από την επισκόπηση των δεδομένων ακολουθεί η προεπεξεργασία τους. Η διαδικασίες αυτού τα σταδίου επιλέγονται με βάση της πληροφορίες που έχουν αντληθεί από την επισκόπηση και

**Αφαίρεση Σταθερών και Σχεδόν Σταθερών Χαρακτηριστικών**

Αφαίρεση των σταθερών και των σχεδόν σταθερών (>=99% σταθερών) χαρακτηριστικών του training set, τόσο από τα training, όσο και τα testing δεδομένα. Αυτό το βήμα πρέπει να γίνει καθώς τα σταθερά χαρακτηριστικά δεν προσφέρουν καμία πληροφορία στο μοντέλο, αυξάνοντας μόνο τις διαστάσεις του προβλήματος.

In [102]:
quasi_constant_thresh = 0.99

quasi_constant_feats = []
for feat in df_train[data_columns].columns:

    top_val_freq = df_train[feat].value_counts(normalize=True, dropna=False).values[0]

    if top_val_freq >= quasi_constant_thresh:
        quasi_constant_feats.append(feat)

print(f"\nConstant / Quasi-Constant (>= {quasi_constant_thresh*100}% same value) "
      f"Feature list:")
print(quasi_constant_feats)
print(f"\nRemoving {len(quasi_constant_feats)} Constant / Quasi-Constant Features...")

df_train_const_clean = df_train.drop(columns=quasi_constant_feats)
df_test_const_clean = df_test.drop(columns=quasi_constant_feats)

print(f'\nOriginal Train shape: {df_train.shape}')
print(f'Original Test shape: {df_test.shape}')

print(f'\nTrain shape after feature removal: {df_train_const_clean.shape}')
print(f'Test shape after feature removal: {df_test_const_clean.shape}\n')


Constant / Quasi-Constant (>= 99.0% same value) Feature list:
['Feat 68', 'Feat 188', 'Feat 191', 'Feat 287', 'Feat 292', 'Feat 388']

Removing 6 Constant / Quasi-Constant Features...

Original Train shape: (1254, 475)
Original Test shape: (313, 474)

Train shape after feature removal: (1254, 469)
Test shape after feature removal: (313, 468)



**Αφαίρεση Χαρακτηριστικών με Μεγάλο Ποσοστό Ελλειπουσών Τιμών**

Χαρακτηριστικά του training set με μεγάλο ποσοστό missing τιμών (>=80%) πρέπει να αφαιρεθούν από το training και testing set γιατί δεν είναι εφικτό να συμπληρωθούν τα κενά χωρίς βλάβη της γενικότητας και η μείωση των διαστάσεων προσφέρει στην συγκεκριμένη περίπτωση μεγαλύτερη αξία. Προτού όμως απομακρυνθούν αυτές οι στήλες είναι απαραίτητο να προσδιοριστεί εάν αυτές οι missing τιμές συσχετίζονται περισσότερο με μία από τις δύο κλάσεις, που σημαίνει ότι αυτό το υψηλό ποσοστό missing προσφέρει χρήσιμες για το μοντέλο πληροφορίες. Αυτή η κατηγορία missing data ονομάζεται **MNAR** και αναλύεται παρακάτω στην υποενότητα ***Διαχείρηση Ελλειπουσών Τιμών***. Εάν το feature δεν ανήκει σε αυτή την κατηγορία, τότε μπορεί ελεύθερα να αφαιρεθεί από τα δεδομένα.

Εύρεση των χαρακτηριστικών με missing data >=80% στο training set. Έλεγχος κάθε εντοπισμένου χαρακτηριστικού για το αν τα missing δεδομένα του ανήκουν στην κατηγορία NMAR συγκρίνοντας τα ποσοστά missing data μεταξύ των κλάσεων. Αν η διαφορά μεταξύ των δύο ποσοστών είναι <20%, τότε το χαρακτηριστικό αφαιρείται από τα training και testing δεδομένα, διαφορετικά διατηρείται.

In [103]:
missing_thresh = 0.80
missing_diff_thresh = 0.10

limit = missing_thresh * len(df_train_const_clean)
high_missing_feats = df_train_const_clean.columns[df_train_const_clean.isnull().sum() > limit].tolist()

print(f"\nMissing >={missing_thresh*100}% of values Feature list:")
print(high_missing_feats)


print('\n' + 10 * "-" + " MNAR Check " + 10 * "-" + '\n')

idx_0 = df_train_const_clean['y'] == 0
idx_1 = df_train_const_clean['y'] == 1

suspicious_feats = []

for feat in high_missing_feats:

    missing_rate_0 = df_train_const_clean.loc[idx_0, feat].isnull().mean()
    missing_rate_1 = df_train_const_clean.loc[idx_1, feat].isnull().mean()

    diff = missing_rate_1 - missing_rate_0
    print(f"{feat:<8}: Missing Rate Class 0(-) = {missing_rate_0:>6.2%} | Class 1(+) = {missing_rate_1:>6.2%} "
          f"| Diff = {diff:>7.2%}")

    if diff > missing_diff_thresh:
        suspicious_feats.append(feat)

print(f"\nSummary: Possible MNAR (Diff >= {missing_diff_thresh*100}%) -> ")
print(suspicious_feats)

print('\n' + 32 * "-")


feats_to_drop = [feat for feat in high_missing_feats if feat not in suspicious_feats]

print(f"\nRemoving {len(feats_to_drop)} features:")
print(feats_to_drop)

df_train_missing_clean = df_train_const_clean.drop(columns=feats_to_drop)
df_test_missing_clean = df_test_const_clean.drop(columns=feats_to_drop)

print(f'\nCurrent Train shape: {df_train_const_clean.shape}')
print(f'Current Test shape: {df_test_const_clean.shape}')

print(f'\nTrain shape after feature removal: {df_train_missing_clean.shape}')
print(f'Test shape after feature removal: {df_test_missing_clean.shape}\n')


Missing >=80.0% of values Feature list:
['Feat 79', 'Feat 148', 'Feat 149', 'Feat 202', 'Feat 247', 'Feat 248', 'Feat 303', 'Feat 401']

---------- MNAR Check ----------

Feat 79 : Missing Rate Class 0(-) = 85.56% | Class 1(+) = 90.48% | Diff =   4.92%
Feat 148: Missing Rate Class 0(-) = 91.79% | Class 1(+) = 94.05% | Diff =   2.25%
Feat 149: Missing Rate Class 0(-) = 91.79% | Class 1(+) = 94.05% | Diff =   2.25%
Feat 202: Missing Rate Class 0(-) = 85.56% | Class 1(+) = 90.48% | Diff =   4.92%
Feat 247: Missing Rate Class 0(-) = 91.79% | Class 1(+) = 94.05% | Diff =   2.25%
Feat 248: Missing Rate Class 0(-) = 91.79% | Class 1(+) = 94.05% | Diff =   2.25%
Feat 303: Missing Rate Class 0(-) = 85.56% | Class 1(+) = 90.48% | Diff =   4.92%
Feat 401: Missing Rate Class 0(-) = 85.56% | Class 1(+) = 90.48% | Diff =   4.92%

Summary: Possible MNAR (Diff >= 10.0%) -> 
[]

--------------------------------

Removing 8 features:
['Feat 79', 'Feat 148', 'Feat 149', 'Feat 202', 'Feat 247', 'Feat 248

Από τα παραπάνω αποτελέσματα γίνεται ξεκάθαρο ότι τουλάχιστον από τα χαρακτηριστικά με μεγαλύτερο ποσοστό από 80% missing data, κανένα δεν διαθέτει MNAR ελλειπείς τιμές, οπότε αυτά μπορούν να αφαιρεθούν άφοβα.


**Διαχείρηση Ελλειπουσών Τιμών**

Προτού εφαρμοστεί η κατάλληλη τεχνική διαχείρησης των missing values, είναι αναγκαίο να προσδιοριστεί η φύση των δεδομένων που λείπουν. Υπάρχουν λοιπόν τρεις κατηγορίες:

*   **Missing Completely At Random (MCAR):** Σε αυτήν την περίπτωση, η πίθανότητα έλλειψης των δεδομένων είναι ανεξάρτητη από τις αντίστοιχες τιμές (παρατηρούμενες ή μη) των χαρακτηριστικών. Για παράδειγμα, στην παρούσα εφαρμογή θα μπορούσε να ήταν missing τιμές λόγω στιγμιαίων διακοπών ρεύματος στους αισθητήρες, κάτι που οφείλεται σε έναν εξωγενή και τελείως τυχαίο παράγοντα.

*   **Missing At Random (MAR):** Εδώ οι missing τιμές χαρακτηριστικών οφείλονται στις μετρούμενες τιμές άλλων χαρακτηριστικών. Κάτι τέτοιο ισχύει όταν, για παράδειγμα, ένας αισθήτηρας σταματάει να καταγράφει τιμές (άρα missing value), όταν ένας άλλος αισθητήρας καταγράφει πάρα πολύ ψηλή τιμή ενός άλλου μεγέθους. Σε αυτήν την περίπτωση, τα ελλειπή δεδομένα δεν οφείλονται στην ίδια την τιμή που κανονικά θα παρατηρούταν, αλλά σε άλλα χαρακτηριστικά για τα οποία έχει καταγραφεί τιμή.

*   **Missing Not At Random (MNAR):** Η έλλειψη τιμής οφείλεται στην ίδια τιμή που λείπει. Εδώ η αιτία δεν είναι καθόλου τυχαία, αλλά εξαρτάται από την ίδια την φύση του μετρούμενου μεγέθους. Στην συγκεκριμένη εφαρμογή, θα μπορούσε να υπάρχει αυτός ο τύπος, εάν ένας αισθητήρας παύει να λειτουργεί, όταν η μετρούμενη τιμή ξεπερνάει ένα μέγεθος.

Από αυτές τις τρείς κατηγορίες, η πιο διαχειρήσιμη είναι η πρώτη, καθώς είναι εντελώς τυχαία, ενώ πιο επικίνδυνη είναι η τρίτη, διότι μπορεί να φανερώνει πληροφορίες για τις τάξεις που προσπαθούμε να διαχωρίσουμε. Κάθε τύπος απαιτεί διαφορετική τεχνική αντιμετώπισης, επομένως πρέπει να προσδιοριστεί ο τύπος των ελλειπουσών τιμών.

In [ ]:
# 1. Εύρεση των στηλών που έχουν όντως κενά
cols_with_missing = df_train.columns[df_train.isnull().any()].tolist()
print(f"Πλήθος στηλών με Missing Values: {len(cols_with_missing)} από {df_train.shape[1]}")

# --- Α. Missing Values Per Feature (Ποσοτικά) ---
missing_counts = df_train[cols_with_missing].isnull().sum().sort_values(ascending=False)
missing_percentages = (missing_counts / len(df_train)) * 100

print("\n--- Top 10 Features με τα περισσότερα κενά ---")
print(pd.concat([missing_counts, missing_percentages.round(2)], axis=1, keys=['Count', 'Percent (%)']).head(30))

# --- Β. Έλεγχος για MNAR (Συσχέτιση Missingness με την Κλάση y) ---
# Δημιουργούμε έναν πίνακα όπου 1 = Λείπει, 0 = Υπάρχει
missing_indicator = df_train[cols_with_missing].isnull().astype(int)

# Προσθέτουμε την κλάση y για να δούμε τη συσχέτιση
missing_indicator['y'] = df_train['y']

# Υπολογίζουμε τη συσχέτιση (correlation) του να λείπει μια τιμή με το να είναι Class 1
correlations_with_target = missing_indicator.corr()['y'].drop('y')
top_mnar_suspects = correlations_with_target.abs().sort_values(ascending=False).head(30)

print("\n--- Top 10 Features που η απουσία τους σχετίζεται με την Κλάση (MNAR check) ---")
print("Υψηλή θετική συσχέτιση σημαίνει: Όταν λείπει η τιμή, είναι πιθανότερο να είναι Αστοχία (Class 1)")
print(top_mnar_suspects)

# --- Γ. Οπτικοποίηση (Heatmap Συσχετίσεων) ---
# Αν δεν είναι πάρα πολλές οι στήλες, κάνουμε heatmap για να δούμε αν λείπουν "πακέτο"
if len(cols_with_missing) < 50:
    plt.figure(figsize=(12, 8))
    sns.heatmap(missing_indicator.drop('y', axis=1).corr(), cmap='coolwarm', vmin=-1, vmax=1)
    plt.title("Correlation Heatmap of Missing Values (Ποια λείπουν μαζί;)")
    plt.show()
else:
    print("\n(Το Heatmap παραλείπεται γιατί οι στήλες με κενά είναι πάρα πολλές για να φανούν καθαρά)")

Πλήθος στηλών με Missing Values: 418 από 475

--- Top 10 Features με τα περισσότερα κενά ---
          Count  Percent (%)
Feat 148   1153        91.95
Feat 248   1153        91.95
Feat 149   1153        91.95
Feat 247   1153        91.95
Feat 401   1077        85.89
Feat 79    1077        85.89
Feat 303   1077        85.89
Feat 202   1077        85.89
Feat 213    818        65.23
Feat 316    818        65.23
Feat 212    818        65.23
Feat 214    818        65.23
Feat 413    818        65.23
Feat 317    818        65.23
Feat 315    818        65.23
Feat 412    818        65.23
Feat 411    818        65.23
Feat 103    818        65.23
Feat 102    818        65.23
Feat 104    818        65.23
Feat 465    767        61.16
Feat 462    767        61.16
Feat 464    767        61.16
Feat 463    767        61.16
Feat 290    610        48.64
Feat 66     610        48.64
Feat 67     610        48.64
Feat 291    610        48.64
Feat 414    569        45.37
Feat 105    569        45.37

--- Top

### **b.**

### **c.**

---

## **Μέρος 2ο – Ταξινόμηση σε Υπερφασματική Εικόνα**

### **a.**

### **b.**

### **c.**